In [1]:
import numpy as np
import scipy.io as sio

def load_and_preprocess_crcns_v1v2(mat_file_path, bin_size_ms=100, min_firing_rate=0.5):
    """
    Loads and preprocesses the CRCNS v1v2-1 dataset into a 3D tensor structure 
    (Trials, Timepoints, Neurons) and computes trial-to-trial PSTH residuals.
    
    Parameters:
    - mat_file_path: Path to the downloaded CRCNS .mat session file.
    - bin_size_ms: Spike counting bin size in milliseconds (default: 100ms as per Semedo et al., 2019).
    - min_firing_rate: Minimum average firing rate threshold to keep a neuron.
    
    Returns:
    - X_tensor: 3D numpy array for Source V1 (Trials, Timepoints, Neurons)
    - Y_tensor: 3D numpy array for Target V2 (Trials, Timepoints, Neurons)
    """
    print(f"Loading dataset from {mat_file_path}...")
    mat_data = sio.loadmat(mat_file_path, squeeze_me=True, struct_as_dict=False)
    
    raw_v1_trials = mat_data.get('v1_trials', None) 
    raw_v2_trials = mat_data.get('v2_trials', None)
    
    if raw_v1_trials is None or raw_v2_trials is None:
        raise ValueError("Could not locate expected trial keys in the .mat file. Check structure via sio.whosmat().")

    # 1. Filter out neurons with mean firing rate below the threshold across all trials
    v1_mean_fr = np.mean(raw_v1_trials, axis=(0, 1)) / (bin_size_ms / 1000.0)
    v2_mean_fr = np.mean(raw_v2_trials, axis=(0, 1)) / (bin_size_ms / 1000.0)
    
    v1_valid_idx = v1_mean_fr >= min_firing_rate
    v2_valid_idx = v2_mean_fr >= min_firing_rate
    
    X_filtered = raw_v1_trials[:, :, v1_valid_idx]
    Y_filtered = raw_v2_trials[:, :, v2_valid_idx]
    
    print(f"Retained V1 neurons: {np.sum(v1_valid_idx)} / {len(v1_valid_idx)}")
    print(f"Retained V2 neurons: {np.sum(v2_valid_idx)} / {len(v2_valid_idx)}")

    # 2. Compute and subtract the Peri-Stimulus Time Histogram (PSTH) per condition
    psth_v1 = np.mean(X_filtered, axis=0) 
    psth_v2 = np.mean(Y_filtered, axis=0)
    
    X_residuals = X_filtered - psth_v1[np.newaxis, :, :]
    Y_residuals = Y_filtered - psth_v2[np.newaxis, :, :]
    
    print(f"Processed 3D Tensor Shapes -> X_residuals (Source V1): {X_residuals.shape}, Y_residuals (Target V2): {Y_residuals.shape}")
    
    return X_residuals, Y_residuals

def extract_window_from_tensor(X_tensor, Y_tensor, window_start, window_end):
    """
    Safely extracts a temporal window across all trials without flattening issues.
    
    X_tensor: (Trials, Timepoints, Neurons)
    Returns flattened matrices for regression fitting within that specific window: 
    X_window: (Trials * Window_Timepoints, Neurons)
    """
    x_slice = X_tensor[:, window_start:window_end, :]
    y_slice = Y_tensor[:, window_start:window_end, :]
    
    X_win_flat = x_slice.reshape(-1, X_tensor.shape[2])
    Y_win_flat = y_slice.reshape(-1, Y_tensor.shape[2])
    
    return X_win_flat, Y_win_flat

In [2]:
import numpy as np
from sklearn.model_selection import KFold

class CrossValidatedPreprocessor:
    """
    Handles trial-wise cross-validation and condition-specific PSTH residualization 
    to prevent information leakage during model training and evaluation.
    """
    def __init__(self, X_trials, Y_trials, conditions, n_splits=10):
        self.X = X_trials
        self.Y = Y_trials
        self.conditions = np.array(conditions)
        self.n_splits = n_splits

    def get_trial_splits(self):
        """
        Generates trial indices for cross-validation, ensuring entire trials 
        (and all their timepoints) remain together in either train or test sets.
        """
        kf = KFold(n_splits=self.n_splits, shuffle=True, random_state=42)
        return list(kf.split(self.X))

    def preprocess_fold(self, train_idx, test_idx):
        """
        Computes condition-specific PSTHs exclusively on training trials, 
        then residualizes both training and test trials to prevent information leakage.
        """
        X_train_raw, X_test_raw = self.X[train_idx], self.X[test_idx]
        Y_train_raw, Y_test_raw = self.Y[train_idx], self.Y[test_idx]
        
        cond_train = self.conditions[train_idx]
        cond_test = self.conditions[test_idx]
        
        unique_conditions = np.unique(self.conditions)
        
        X_train_res = np.empty_like(X_train_raw)
        X_test_res = np.empty_like(X_test_raw)
        Y_train_res = np.empty_like(Y_train_raw)
        Y_test_res = np.empty_like(Y_test_raw)
        
        for cond in unique_conditions:
            train_cond_mask = (cond_train == cond)
            
            if np.sum(train_cond_mask) > 0:
                psth_v1_cond = np.mean(X_train_raw[train_cond_mask], axis=0)
                psth_v2_cond = np.mean(Y_train_raw[train_cond_mask], axis=0)
            else:
                psth_v1_cond = np.mean(X_train_raw, axis=0)
                psth_v2_cond = np.mean(Y_train_raw, axis=0)
            
            train_target_mask = (cond_train == cond)
            if np.sum(train_target_mask) > 0:
                X_train_res[train_target_mask] = X_train_raw[train_target_mask] - psth_v1_cond[np.newaxis, :, :]
                Y_train_res[train_target_mask] = Y_train_raw[train_target_mask] - psth_v2_cond[np.newaxis, :, :]
                
            test_target_mask = (cond_test == cond)
            if np.sum(test_target_mask) > 0:
                X_test_res[test_target_mask] = X_test_raw[test_target_mask] - psth_v1_cond[np.newaxis, :, :]
                Y_test_res[test_target_mask] = Y_test_raw[test_target_mask] - psth_v2_cond[np.newaxis, :, :]
                
        return X_train_res, Y_train_res, X_test_res, Y_test_res

In [3]:
import numpy as np

class SemedoBaselineEngine:
    """
    Implements the baseline Reduced-Rank Regression (RRR) engine using stable 
    linear solvers and evaluates performance on held-out trial folds.
    """
    def __init__(self, l2_reg=1e-3):
        self.l2_reg = l2_reg

    def fit_and_evaluate(self, X_train_res, Y_train_res, X_test_res, Y_test_res, rank=2):
        """
        Fits ridge regression and RRR on training data, then evaluates out-of-sample 
        predictive performance (R-squared) on test data.
        """
        p_source = X_train_res.shape[1]
        
        XtX = X_train_res.T @ X_train_res
        ridge_matrix = self.l2_reg * np.eye(p_source)
        
        target_term = X_train_res.T @ Y_train_res
        B_ols = np.linalg.solve(XtX + ridge_matrix, target_term)
        
        Y_ols_train = X_train_res @ B_ols
        _, _, Vt = np.linalg.svd(Y_ols_train, full_matrices=False)
        V = Vt.T
        
        V_m = V[:, :rank]
        B_rrr = B_ols @ V_m @ V_m.T
        B_bar = B_ols @ V_m
        
        Y_pred_test = X_test_res @ B_rrr
        
        residual_variance = np.var(Y_test_res - Y_pred_test, axis=0)
        total_variance = np.var(Y_test_res, axis=0)
        r2_per_neuron = 1.0 - (residual_variance / (total_variance + 1e-10))
        mean_test_r2 = np.mean(r2_per_neuron)
        
        return {
            "B_ols": B_ols,
            "B_rrr": B_rrr,
            "B_bar": B_bar,
            "test_r2": mean_test_r2,
            "r2_per_neuron": r2_per_neuron
        }

In [6]:
import numpy as np

class RobustDynamicSubspaceTracker:
    """
    Tracks rolling time-window communication subspace rotations with trial-level 
    bootstrapping for uncertainty quantification and rank-deficiency safeguards.
    """
    def __init__(self, X_tensor, Y_tensor, l2_reg=1e-3):
        self.X_tensor = X_tensor
        self.Y_tensor = Y_tensor
        self.l2_reg = l2_reg
        self.n_trials, self.n_timepoints, self.p_source = X_tensor.shape
        _, _, self.q_target = Y_tensor.shape

    def _fit_window_basis(self, X_win, Y_win, rank):
        XtX = X_win.T @ X_win
        ridge_matrix = self.l2_reg * np.eye(self.p_source)
        
        target_term = X_win.T @ Y_win
        B_ols = np.linalg.solve(XtX + ridge_matrix, target_term)
        
        Y_ols = X_win @ B_ols
        _, S, Vt = np.linalg.svd(Y_ols, full_matrices=False)
        
        effective_rank = min(rank, np.sum(S > 1e-5))
        V_reduced = Vt[:effective_rank, :].T
        
        B_bar = B_ols @ V_reduced
        Q, _ = np.linalg.qr(B_bar)
        return Q

    def compute_window_bases(self, window_size=20, step_size=10, rank=2, trial_indices=None):
        if trial_indices is None:
            trial_indices = np.arange(self.n_trials)
            
        subspace_bases = []
        window_centers = []
        
        for start_t in range(0, self.n_timepoints - window_size + 1, step_size):
            end_t = start_t + window_size
            
            x_slice = self.X_tensor[trial_indices, start_t:end_t, :]
            y_slice = self.Y_tensor[trial_indices, start_t:end_t, :]
            
            X_win = x_slice.reshape(-1, self.p_source)
            Y_win = y_slice.reshape(-1, self.q_target)
            
            Q_t = self._fit_window_basis(X_win, Y_win, rank=rank)
            subspace_bases.append(Q_t)
            window_centers.append((start_t + end_t) / 2.0)
            
        return np.array(window_centers), subspace_bases

    def _calculate_rotation_angles(self, bases):
        rotation_angles = []
        for i in range(len(bases) - 1):
            B1, B2 = bases[i], bases[i+1]
            cross_overlap = B1.T @ B2
            singular_values = np.linalg.svd(cross_overlap, compute_uv=False)
            clipped_sv = np.clip(singular_values, -1.0, 1.0)
            angles_deg = np.degrees(np.arccos(clipped_sv))
            rotation_angles.append(np.max(angles_deg))
        return np.array(rotation_angles)

    def track_dynamics_with_bootstrapping(self, window_size=20, step_size=10, rank=2, n_bootstraps=50):
        print(f"Running bootstrap dynamic tracking ({n_bootstraps} resamples)...")
        bootstrapped_angles = []
        centers = None
        
        for b in range(n_bootstraps):
            resampled_trials = np.random.choice(self.n_trials, size=self.n_trials, replace=True)
            centers, bases = self.compute_window_bases(window_size, step_size, rank, trial_indices=resampled_trials)
            angles = self._calculate_rotation_angles(bases)
            bootstrapped_angles.append(angles)
            
        bootstrapped_angles = np.array(bootstrapped_angles)
        
        mean_angles = np.mean(bootstrapped_angles, axis=0)
        ci_lower = np.percentile(bootstrapped_angles, 2.5, axis=0)
        ci_upper = np.percentile(bootstrapped_angles, 97.5, axis=0)
        
        return {
            "window_centers": centers[1:],
            "mean_rotation_angles_deg": mean_angles,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
            "raw_bootstraps": bootstrapped_angles
        }
        
       

In [7]:
import numpy as np

class CorrectedMultiplexedAnalyzer:
    """
    Partitions target populations using unresidualized spike metrics or biological 
    metadata, then measures subspace overlap via principal angles.
    """
    def __init__(self, X_flat, Y_flat, Y_raw_tensor):
        self.X = X_flat
        self.Y_res = Y_flat
        self.raw_mean_fr = np.mean(Y_raw_tensor, axis=(0, 1))
        self.p_source = X_flat.shape[1]
        self.q_target = Y_flat.shape[1]

    def partition_by_raw_firing_rate(self):
        median_fr = np.median(self.raw_mean_fr)
        group1_idx = self.raw_mean_fr <= median_fr
        group2_idx = self.raw_mean_fr > median_fr
        
        Y_sub1 = self.Y_res[:, group1_idx]
        Y_sub2 = self.Y_res[:, group2_idx]
        return Y_sub1, Y_sub2

    def compute_subspace_overlap(self, Y_sub1, Y_sub2, rank=2, l2_reg=1e-3):
        XtX = self.X.T @ self.X
        ridge_matrix = l2_reg * np.eye(self.p_source)
        inv_XtX = np.linalg.solve(XtX + ridge_matrix, np.eye(self.p_source))
        
        B_ols_1 = inv_XtX @ (self.X.T @ Y_sub1)
        _, _, Vt1 = np.linalg.svd(self.X @ B_ols_1, full_matrices=False)
        B1 = B_ols_1 @ Vt1[:rank, :].T
        Q1, _ = np.linalg.qr(B1)
        
        B_ols_2 = inv_XtX @ (self.X.T @ Y_sub2)
        _, _, Vt2 = np.linalg.svd(self.X @ B_ols_2, full_matrices=False)
        B2 = B_ols_2 @ Vt2[:rank, :].T
        Q2, _ = np.linalg.qr(B2)
        
        cross_overlap = Q1.T @ Q2
        singular_values = np.linalg.svd(cross_overlap, compute_uv=False)
        angles_deg = np.degrees(np.arccos(np.clip(singular_values, -1.0, 1.0)))
        
        return angles_deg

In [8]:
import numpy as np

class SubspaceNullValidator:
    """
    Generates surrogate null distributions via trial shuffling to test statistical 
    significance of principal angles and communication subspace geometry.
    """
    def __init__(self, X_tensor, Y_tensor, l2_reg=1e-3):
        self.X_tensor = X_tensor
        self.Y_tensor = Y_tensor
        self.l2_reg = l2_reg

    def generate_shuffled_null_angles(self, window_size=20, step_size=10, rank=2, n_shuffles=100):
        print(f"Generating shuffled null distribution ({n_shuffles} iterations)...")
        n_trials, n_timepoints, p_source = self.X_tensor.shape
        q_target = self.Y_tensor.shape[2]
        
        null_max_angles = []
        
        for _ in range(n_shuffles):
            shuffled_indices = np.random.permutation(n_trials)
            Y_shuffled = self.Y_tensor[shuffled_indices, :, :]
            
            subspace_bases = []
            for start_t in range(0, n_timepoints - window_size + 1, step_size):
                end_t = start_t + window_size
                X_win = self.X_tensor[:, start_t:end_t, :].reshape(-1, p_source)
                Y_win = Y_shuffled[:, start_t:end_t, :].reshape(-1, q_target)
                
                XtX = X_win.T @ X_win
                ridge = self.l2_reg * np.eye(p_source)
                B_ols = np.linalg.solve(XtX + ridge, X_win.T @ Y_win)
                _, _, Vt = np.linalg.svd(X_win @ B_ols, full_matrices=False)
                
                B_bar = B_ols @ Vt[:rank, :].T
                Q, _ = np.linalg.qr(B_bar)
                subspace_bases.append(Q)
                
            run_angles = []
            for i in range(len(subspace_bases) - 1):
                overlap = subspace_bases[i].T @ subspace_bases[i+1]
                sv = np.linalg.svd(overlap, compute_uv=False)
                run_angles.append(np.max(np.degrees(np.arccos(np.clip(sv, -1.0, 1.0)))))
                
            null_max_angles.append(np.mean(run_angles))
            
        return np.array(null_max_angles)

    def evaluate_significance(self, observed_mean_angle, null_distribution):
        p_val = np.sum(null_distribution <= observed_mean_angle) / len(null_distribution)
        return p_val

In [9]:
import numpy as np
from preprocessing import load_and_preprocess_crcns_v1v2
from cv_preprocessor import CrossValidatedPreprocessor
from semedo_baseline import SemedoBaselineEngine
from subspace_tracker import RobustDynamicSubspaceTracker
from multiplexed_analyzer import CorrectedMultiplexedAnalyzer
from subspace_null_validator import SubspaceNullValidator

def run_communication_subspace_pipeline(mat_file_path):
    print("=" * 60)
    print("STARTING COMMUNICATION SUBSPACE PIPELINE")
    print("=" * 60)

    # ---------------------------------------------------------
    # 1. Load and Preprocess Dataset
    # ---------------------------------------------------------
    # Note: Adjust bin_size_ms and min_firing_rate as needed for your session
    X_tensor, Y_tensor = load_and_preprocess_crcns_v1v2(
        mat_file_path=mat_file_path, 
        bin_size_ms=100, 
        min_firing_rate=0.5
    )
    
    n_trials, n_timepoints, p_source = X_tensor.shape
    q_target = Y_tensor.shape[2]
    
    # Placeholder for trial conditions (e.g., stimulus orientation labels).
    # Replace this with your actual condition labels loaded from the session file.
    conditions = np.random.randint(0, 4, size=n_trials)

    # ---------------------------------------------------------
    # 2. Cross-Validation & Residualization Setup
    # ---------------------------------------------------------
    print("\n[Step 2] Initializing Cross-Validated Preprocessor...")
    cv_prep = CrossValidatedPreprocessor(X_tensor, Y_tensor, conditions, n_splits=5)
    splits = cv_prep.get_trial_splits()

    # ---------------------------------------------------------
    # 3. Baseline Reduced-Rank Regression (RRR) & Evaluation
    # ---------------------------------------------------------
    print("\n[Step 3] Running Cross-Validated Semedo RRR Baseline...")
    baseline_engine = SemedoBaselineEngine(l2_reg=1e-3)
    fold_r2_scores = []

    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        # Process fold without data leakage
        X_train_res, Y_train_res, X_test_res, Y_test_res = cv_prep.preprocess_fold(train_idx, test_idx)
        
        # Flatten time and trial dimensions for standard regression fitting: (Trials * Time, Neurons)
        X_train_flat = X_train_res.reshape(-1, p_source)
        Y_train_flat = Y_train_res.reshape(-1, q_target)
        X_test_flat = X_test_res.reshape(-1, p_source)
        Y_test_flat = Y_test_res.reshape(-1, q_target)
        
        # Fit and evaluate model
        results = baseline_engine.fit_and_evaluate(
            X_train_flat, Y_train_flat, X_test_flat, Y_test_flat, rank=2
        )
        fold_r2_scores.append(results["test_r2"])
        print(f"  -> Fold {fold_idx + 1} Test R²: {results['test_r2']:.4f}")

    print(f"Mean Cross-Validated Test R²: {np.mean(fold_r2_scores):.4f}")

    # ---------------------------------------------------------
    # 4. Dynamic Subspace Tracking with Bootstrapping
    # ---------------------------------------------------------
    print("\n[Step 4] Tracking Subspace Dynamics & Uncertainty...")
    # Use full residualized representation or raw tensor across time windows
    tracker = RobustDynamicSubspaceTracker(X_tensor, Y_tensor, l2_reg=1e-3)
    dynamics_results = tracker.track_dynamics_with_bootstrapping(
        window_size=20, step_size=10, rank=2, n_bootstraps=20
    )
    print(f"  -> Tracked {len(dynamics_results['window_centers'])} temporal windows.")
    print(f"  -> Mean Rotation Angle across windows: {np.mean(dynamics_results['mean_rotation_angles_deg']):.2f}°")

    # ---------------------------------------------------------
    # 5. Multiplexed Subpopulation Analysis
    # ---------------------------------------------------------
    print("\n[Step 5] Analyzing Target Subpopulations...")
    # Flatten tensors for static subpopulation comparison
    X_flat_all = X_tensor.reshape(-1, p_source)
    Y_flat_all = Y_tensor.reshape(-1, q_target)
    
    analyzer = CorrectedMultiplexedAnalyzer(X_flat_all, Y_flat_all, Y_tensor)
    Y_sub1, Y_sub2 = analyzer.partition_by_raw_firing_rate()
    subspace_angles = analyzer.compute_subspace_overlap(Y_sub1, Y_sub2, rank=2)
    print(f"  -> Subpopulation Principal Angles: {subspace_angles}")

    # ---------------------------------------------------------
    # 6. Statistical Validation via Surrogate Nulls
    # ---------------------------------------------------------
    print("\n[Step 6] Running Subspace Null Validation...")
    validator = SubspaceNullValidator(X_tensor, Y_tensor, l2_reg=1e-3)
    null_distribution = validator.generate_shuffled_null_angles(
        window_size=20, step_size=10, rank=2, n_shuffles=20
    )
    
    observed_mean_angle = np.mean(dynamics_results["mean_rotation_angles_deg"])
    p_value = validator.evaluate_significance(observed_mean_angle, null_distribution)
    print(f"  -> Observed Mean Angle: {observed_mean_angle:.2f}°")
    print(f"  -> Surrogate Null p-value: {p_value:.4f}")

    print("\n" + "=" * 60)
    print("PIPELINE COMPLETED SUCCESSFULLY")
    print("=" * 60)

if __name__ == "__main__":
    # Example entry point execution
    sample_mat_path = "path_to_crcns_session.mat"
    # run_communication_subspace_pipeline(sample_mat_path)

ModuleNotFoundError: No module named 'preprocessing'